# FL Partitioning Strategies — CircuitNet-N28 Visualization

This notebook applies and visualizes all six partitioning strategies defined in the
NIID-Bench taxonomy to the CircuitNet-N28 dataset metadata.

## Strategies covered

| # | Strategy | Skew type |
|---|----------|-----------|
| 1 | **IID** | None (baseline) |
| 2 | **Quantity-based label imbalance** | Label distribution |
| 3 | **Distribution-based label imbalance (Dirichlet)** | Label distribution |
| 4 | **Noise-based feature imbalance** | Feature distribution |
| 5 | **Synthetic feature imbalance** | Feature distribution |
| 6 | **Quantity skew (Dirichlet)** | Quantity |

The notebook works both when the real dataset is present (`.npy` files under
`../../drc_prediction/training_set/`) and when it is absent — in the latter case
a faithful mock DataFrame is generated from the known CircuitNet-N28 design space.

**Label for partitioning purposes:** `design_name` (the six base RTL designs).  
**Feature axes used:** `utilization` × `clock_ns` (continuous design-space knobs).

In [ ]:
import os
import sys
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.cm as mcm

warnings.filterwarnings('ignore')
plt.rcParams.update({
    'figure.figsize': (14, 5),
    'axes.titlesize': 11,
    'axes.grid': True,
    'grid.color': 'white',
    'grid.linewidth': 0.8,
    'axes.facecolor': '#f5f5f5',
    'figure.facecolor': 'white',
})

# Add partitioning package to path
sys.path.insert(0, os.path.abspath('.'))

from partitioning import (
    IIDPartitioner,
    QuantityLabelPartitioner,
    DirichletLabelPartitioner,
    NoiseFeaturePartitioner,
    SyntheticFeaturePartitioner,
    QuantitySkewPartitioner,
)

print('All partitioners imported successfully.')

## Dataset Loading

Try to load the real CircuitNet-N28 filenames; fall back to mock data if the
dataset is not available locally.

In [ ]:
def parse_sample_name(filename: str) -> dict:
    """Parse a CircuitNet-N28 filename into its design-space components.

    Handles both the raw 7-segment form and the prefixed form used in the
    DRC annotation CSV (e.g. '1-RISCY-a-1-c2-u0.7-m1-p1-f0.npy').
    """
    basename = filename.replace('.npy', '')
    parts = basename.split('-')

    # Strip leading numeric prefix if present (e.g. '1-RISCY-a-...')
    if parts[0].isdigit():
        parts = parts[1:]

    # Design names may contain a hyphen (e.g. 'RISCY-a')
    # Detect by checking whether parts[1] is a single letter (design variant)
    if len(parts) >= 2 and len(parts[1]) == 1 and parts[1].isalpha():
        design_name = parts[0] + '-' + parts[1]
        rest = parts[2:]
    else:
        design_name = parts[0]
        rest = parts[1:]

    if len(rest) < 6:
        raise ValueError(f'Cannot parse filename: {filename}')

    macro_count   = rest[0]
    clock_ns      = float(rest[1][1:])   # remove 'c'
    utilization   = float(rest[2][1:])   # remove 'u'
    macro_placement = rest[3][1:]        # remove 'm'
    power_mesh    = rest[4][1:]          # remove 'p'
    filler        = rest[5][1:]          # remove 'f'

    return {
        'design_name':     design_name,
        'macro_count':     macro_count,
        'clock_ns':        clock_ns,
        'utilization':     utilization,
        'macro_placement': macro_placement,
        'power_mesh':      power_mesh,
        'filler_insertion': filler,
        'filename':        filename,
    }


# Quick smoke-test
ex = parse_sample_name('RISCY-a-1-c2-u0.7-m1-p1-f0.npy')
print('Parse test:', ex)

In [ ]:
from partitioning import LabelTierAssigner

FEATURE_DIR = '../../drc_prediction/training_set/feature/'
LABEL_DIR   = '../../drc_prediction/training_set/label/'
VIOLATION_THRESHOLD = 0.1

feature_exists = os.path.isdir(FEATURE_DIR)
label_exists   = os.path.isdir(LABEL_DIR)

if feature_exists and label_exists:
    files = [f for f in os.listdir(FEATURE_DIR) if f.endswith('.npy')]
    print(f'Real dataset found: {len(files)} samples')
    records = []
    for fname in files:
        try:
            records.append(parse_sample_name(fname))
        except Exception as e:
            print(f'  Skipping {fname}: {e}')
    df = pd.DataFrame(records)

    # Compute per-sample violation rate from label .npy files
    print('Loading label files to compute violation rates...')
    assigner = LabelTierAssigner(threshold=VIOLATION_THRESHOLD)
    rates, tiers = [], []
    for fname in df['filename']:
        lpath = os.path.join(LABEL_DIR, fname)
        try:
            arr = np.load(lpath)
            rate = float(np.mean(arr >= VIOLATION_THRESHOLD))
        except Exception:
            rate = float('nan')
        rates.append(rate)
        tiers.append(assigner.assign_tier(rate) if not np.isnan(rate) else -1)
    df['violation_rate'] = rates
    df['tier'] = tiers
    DATASET_SOURCE = 'real'

else:
    print('Dataset not found locally — generating faithful mock data.')

    rng = np.random.default_rng(0)
    DESIGNS = ['RISCY-a', 'RISCY-b', 'RISCY-c', 'RISCY-d', 'RISCY-e', 'RISCY-f']
    CLOCKS  = [1.0, 1.5, 2.0, 2.5, 3.0, 3.3]
    UTILS   = [0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
    MACROS  = ['1', '2', '3', '4']
    MPL     = ['1', '2', '3']
    PMESH   = ['1', '2']
    FILLER  = ['0', '1']

    N = 10242
    records = []
    for _ in range(N):
        d  = rng.choice(DESIGNS)
        mc = rng.choice(MACROS)
        c  = rng.choice(CLOCKS)
        u  = rng.choice(UTILS)
        mp = rng.choice(MPL)
        pm = rng.choice(PMESH)
        fi = rng.choice(FILLER)
        fname = f'{d}-{mc}-c{c}-u{u}-m{mp}-p{pm}-f{fi}.npy'
        records.append({
            'design_name':      d,
            'macro_count':      mc,
            'clock_ns':         float(c),
            'utilization':      float(u),
            'macro_placement':  mp,
            'power_mesh':       pm,
            'filler_insertion': fi,
            'filename':         fname,
        })
    df = pd.DataFrame(records)

    # Simulate violation_rate correlated with design parameters, then assign tiers
    # higher utilization → more congestion; shorter clock → tighter timing
    rng_vr = np.random.default_rng(7)
    design_offset = {d: rng_vr.uniform(-0.03, 0.03) for d in DESIGNS}
    assigner = LabelTierAssigner(threshold=VIOLATION_THRESHOLD)

    def _sim_rate(row):
        u_effect = 0.12 * (row['utilization'] - 0.4) / 0.5
        c_effect = 0.06 * (1.0 - row['clock_ns'] / 3.3)
        base = 0.04 + u_effect + c_effect + design_offset[row['design_name']]
        return float(np.clip(base + rng_vr.normal(0, 0.015), 0.0, 1.0))

    df['violation_rate'] = df.apply(_sim_rate, axis=1)
    df['tier'] = df['violation_rate'].apply(assigner.assign_tier)
    DATASET_SOURCE = 'mock'

print(f'DataFrame shape: {df.shape}  |  source: {DATASET_SOURCE}')
print(f'Tier distribution:\n{df["tier"].value_counts().sort_index().rename(assigner.TIER_NAMES)}')
df.head(3)

## Shared Visualization Helpers

In [ ]:
# --- Color palettes (pure matplotlib, no seaborn) ---
_TAB10   = mcm.get_cmap('tab10')
_SET2    = mcm.get_cmap('Set2')
PARTY_PALETTE  = [_TAB10(i / 10) for i in range(10)]
DESIGN_PALETTE = [_SET2(i / 8)   for i in range(8)]


def _label_colors(labels):
    """Return a dict mapping each label to a distinct Set2 colour."""
    cmap = mcm.get_cmap('Set2', len(labels))
    return {lbl: cmap(i) for i, lbl in enumerate(sorted(labels))}


def party_label(i: int) -> str:
    return f'Party {i}'


def plot_partition_sizes(partitions, title='Partition sizes', ax=None):
    """Bar chart of samples per party."""
    if ax is None:
        _, ax = plt.subplots(figsize=(6, 4))
    sizes = [len(p) for p in partitions]
    bars = ax.bar(
        [party_label(i) for i in range(len(partitions))],
        sizes,
        color=[PARTY_PALETTE[i % 10] for i in range(len(partitions))],
    )
    ax.set_title(title)
    ax.set_ylabel('Samples')
    ax.set_xlabel('Party')
    for bar, s in zip(bars, sizes):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 5,
            str(s), ha='center', va='bottom', fontsize=8,
        )
    return ax


def plot_label_distribution(partitions, label_col='design_name',
                             title='Label distribution', ax=None):
    """Stacked bar chart of label class counts per party."""
    all_labels = sorted(pd.concat(partitions)[label_col].unique())
    palette = _label_colors(all_labels)

    counts = pd.DataFrame({
        party_label(i): p[label_col].value_counts().reindex(all_labels, fill_value=0)
        for i, p in enumerate(partitions)
    }).T

    if ax is None:
        _, ax = plt.subplots(figsize=(8, 4))

    bottom = np.zeros(len(partitions))
    for lbl in all_labels:
        vals = counts[lbl].values
        ax.bar(counts.index, vals, bottom=bottom,
               color=palette[lbl], label=lbl, alpha=0.87)
        bottom += vals

    ax.set_title(title)
    ax.set_ylabel('Samples')
    ax.set_xlabel('Party')
    ax.legend(title=label_col, bbox_to_anchor=(1.01, 1),
               loc='upper left', fontsize=7)
    ax.tick_params(axis='x', rotation=30)
    return ax


def plot_feature_boxplots(partitions, col, title=None, ax=None):
    """Side-by-side box plots of a continuous feature across parties."""
    if ax is None:
        _, ax = plt.subplots(figsize=(7, 4))
    data_by_party = [p[col].dropna().values for p in partitions]
    bp = ax.boxplot(data_by_party, patch_artist=True, notch=False,
                    medianprops={'color': 'black', 'linewidth': 1.5})
    for patch, color in zip(bp['boxes'], PARTY_PALETTE):
        patch.set_facecolor(color)
        patch.set_alpha(0.72)
    ax.set_xticks(range(1, len(partitions) + 1))
    ax.set_xticklabels([party_label(i) for i in range(len(partitions))], rotation=30)
    ax.set_title(title or col)
    ax.set_ylabel(col)
    return ax


def show_summary(partitions, label_col='design_name',
                 feature_cols=('utilization', 'clock_ns'), method_name='Method'):
    """4-panel summary: sizes | label dist | feature boxplot × 2."""
    fig, axes = plt.subplots(1, 4, figsize=(22, 5))
    fig.suptitle(method_name, fontsize=13, fontweight='bold')

    plot_partition_sizes(partitions, title='Partition sizes', ax=axes[0])
    plot_label_distribution(partitions, label_col=label_col,
                            title=f'{label_col} distribution', ax=axes[1])
    for ax, fc in zip(axes[2:], feature_cols):
        if fc in partitions[0].columns:
            plot_feature_boxplots(partitions, col=fc, title=f'{fc} per party', ax=ax)

    plt.tight_layout()
    plt.show()


N_PARTIES = 5  # number of federated parties used across all experiments
print(f'Visualization helpers ready. N_PARTIES = {N_PARTIES}')

---
## Strategy 1 — IID Partitioning

Each party receives an equal share of samples mirroring the global distribution
across all categorical design axes.  This is the baseline: P(X_i) = P(X_j) and
P(y_i) = P(y_j) for all parties i, j.  Implemented as stratified round-robin
within every (design × macro × placement × power-mesh × filler) stratum.

In [ ]:
iid_parts = IIDPartitioner(n_partitions=N_PARTIES, seed=42).partition(df)

print('IID sizes:', [len(p) for p in iid_parts])
show_summary(iid_parts, method_name='Strategy 1 — IID Partitioning')

---
## Strategy 2 — Quantity-based Label Imbalance

Each party is assigned exactly **k** of the available `design_name` classes.
For every class its samples are split equally among the parties that own it —
no sample overlap across parties.  Lower k → stronger label skew.  We compare
k = 2 (each party sees only 2 out of 6 designs) and k = 4.

In [ ]:
TIER_NAMES = assigner.TIER_NAMES  # {0: 'clean', 1: 'low', 2: 'medium', 3: 'high'}

for k in [2, 3]:
    parts = QuantityLabelPartitioner(
        n_partitions=N_PARTIES, n_labels_per_party=k, label_col='tier', seed=42
    ).partition(df)
    print(f'k={k} sizes:', [len(p) for p in parts])
    show_summary(parts, label_col='tier',
                 method_name=f'Strategy 2 — Quantity-based Label Imbalance  (k={k} tiers/party)')

# Which tiers does each party hold for k=2?
k2_parts = QuantityLabelPartitioner(
    n_partitions=N_PARTIES, n_labels_per_party=2, label_col='tier', seed=42
).partition(df)
print('\nTiers owned per party (k=2):')
for i, p in enumerate(k2_parts):
    owned = sorted(p['tier'].unique())
    names = [TIER_NAMES.get(t, t) for t in owned]
    print(f'  Party {i}: tiers {owned} {names}  ({len(p)} samples)')

---
## Strategy 3 — Distribution-based Label Imbalance (Dirichlet)

For each label class a Dirichlet(α) draw determines the fraction of that
class's samples allocated to each party.  Smaller α → more concentrated
distribution → stronger skew.  α → ∞ recovers IID.

We sweep α ∈ {0.1, 0.5, 5.0} to show the full range of skew.

In [ ]:
for alpha in [0.1, 0.5, 5.0]:
    parts = DirichletLabelPartitioner(
        n_partitions=N_PARTIES, alpha=alpha, label_col='tier', seed=42
    ).partition(df)
    print(f'α={alpha} sizes:', [len(p) for p in parts])
    show_summary(parts, label_col='tier',
                 method_name=f'Strategy 3 — Dirichlet Label Imbalance  (α={alpha})')

In [ ]:
# Heat-map: tier proportion per party across alpha values (pure matplotlib)
all_tiers = sorted(df['tier'].unique())
tier_labels = [f"{t}-{assigner.TIER_NAMES.get(t,'?')}" for t in all_tiers]
alphas = [0.1, 0.5, 5.0]

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
fig.suptitle('Strategy 3 — Dirichlet: tier proportion per party',
             fontsize=12, fontweight='bold')

for ax, alpha in zip(axes, alphas):
    parts = DirichletLabelPartitioner(
        n_partitions=N_PARTIES, alpha=alpha, label_col='tier', seed=42
    ).partition(df)

    mat = np.array([
        p['tier'].value_counts().reindex(all_tiers, fill_value=0).values / max(len(p), 1)
        for p in parts
    ])  # shape: (N_PARTIES, n_tiers)

    im = ax.imshow(mat, aspect='auto', cmap='YlOrRd', vmin=0.0, vmax=1.0)

    for r in range(N_PARTIES):
        for c in range(len(all_tiers)):
            val = mat[r, c]
            txt_color = 'white' if val > 0.55 else 'black'
            ax.text(c, r, f'{val:.2f}', ha='center', va='center',
                    fontsize=8, color=txt_color)

    ax.set_xticks(range(len(all_tiers)))
    ax.set_xticklabels(tier_labels, rotation=30, ha='right', fontsize=8)
    ax.set_yticks(range(N_PARTIES))
    ax.set_yticklabels([f'P{i}' for i in range(N_PARTIES)])
    ax.set_title(f'α = {alpha}')
    ax.set_xlabel('Tier')
    ax.set_ylabel('Party')

plt.colorbar(im, ax=axes[-1], fraction=0.04, label='Proportion')
plt.tight_layout()
plt.show()

---
## Strategy 4 — Noise-based Feature Imbalance

The dataset is first split into equal-sized IID partitions (same label
distribution), then each party is tagged with a Gaussian noise std that
increases linearly from Party 0 to Party N−1.  Actual noise injection on
the `.npy` feature tensors happens at training time; the partitioner adds a
`noise_std` column so each DataLoader knows how much noise to apply.

This creates P(X_i) ≠ P(X_j) while keeping P(y_i | X_i) ≈ P(y_j | X_j).

In [ ]:
noise_parts = NoiseFeaturePartitioner(
    n_partitions=N_PARTIES,
    noise_std_min=0.0,
    noise_std_max=0.5,
    seed=42,
).partition(df)

print('Noise-feature sizes:', [len(p) for p in noise_parts])
print('Assigned noise_std per party:',
      [round(p['noise_std'].iloc[0], 3) for p in noise_parts])

# 4-panel: sizes | label dist | utilization | noise_std bar
fig, axes = plt.subplots(1, 4, figsize=(22, 5))
fig.suptitle('Strategy 4 — Noise-based Feature Imbalance', fontsize=13, fontweight='bold')

plot_partition_sizes(noise_parts, title='Partition sizes', ax=axes[0])
plot_label_distribution(noise_parts, label_col='design_name',
                        title='design_name distribution', ax=axes[1])
plot_feature_boxplots(noise_parts, col='utilization',
                      title='utilization per party', ax=axes[2])

# noise_std assigned per party
stds = [p['noise_std'].iloc[0] for p in noise_parts]
axes[3].bar([f'Party {i}' for i in range(N_PARTIES)], stds,
            color=[PARTY_PALETTE[i % 10] for i in range(N_PARTIES)])
axes[3].set_title('Assigned noise_std per party')
axes[3].set_ylabel('noise_std (σ)')
axes[3].tick_params(axis='x', rotation=30)
for i, v in enumerate(stds):
    axes[3].text(i, v + 0.005, f'{v:.3f}', ha='center', fontsize=8)

plt.tight_layout()
plt.show()

---
## Strategy 5 — Synthetic Feature Imbalance

Inspired by the NIID-Bench "cube division": the 2-D design-parameter space
(utilization × clock_ns) is divided into a quantile grid of cells, one cell
per party.  Each party sees a distinct region of the feature space while the
label (design_name) distribution within each cell remains roughly balanced.

Grid layout is the closest integer factorisation to a square
(e.g. 5 parties → 2×3 grid, last cell merged).

In [ ]:
synth_parts = SyntheticFeaturePartitioner(
    n_partitions=N_PARTIES,
    feature_col_x='utilization',
    feature_col_y='clock_ns',
    seed=42,
).partition(df)

print('Synthetic-feature sizes:', [len(p) for p in synth_parts])
show_summary(synth_parts, method_name='Strategy 5 — Synthetic Feature Imbalance')

In [ ]:
# 2-D scatter: each party coloured differently in (utilization, clock_ns) space
fig, ax = plt.subplots(figsize=(8, 6))
for i, p in enumerate(synth_parts):
    ax.scatter(
        p['utilization'], p['clock_ns'],
        s=4, alpha=0.4, color=PARTY_PALETTE[i % 10], label=f'Party {i}'
    )
ax.set_xlabel('utilization')
ax.set_ylabel('clock_ns')
ax.set_title('Strategy 5 — Feature-space grid assignment')
ax.legend(markerscale=4, loc='upper right')
plt.tight_layout()
plt.show()

---
## Strategy 6 — Quantity Skew (Dirichlet)

The total dataset is randomly shuffled, then each party's share is determined
by a Dirichlet(α) draw.  Label composition within each party mirrors the
global distribution (no label skew), so only the *amount* of data varies.

Lower α → more extreme size imbalance (one party may hold the majority of
the data while others receive very little).

In [ ]:
for alpha in [0.1, 0.5, 5.0]:
    parts = QuantitySkewPartitioner(
        n_partitions=N_PARTIES, alpha=alpha, seed=42
    ).partition(df)
    print(f'α={alpha} sizes:', [len(p) for p in parts])
    show_summary(parts, method_name=f'Strategy 6 — Quantity Skew  (α={alpha})')

---
## Summary — Cross-strategy Comparison

The table and chart below compare all strategies on two axes:

* **Size imbalance** — standard deviation of partition sizes (higher = more unequal).
* **Label entropy imbalance** — mean absolute deviation of per-party label entropy
  from the global entropy (higher = more label skew).

In [ ]:
from scipy.stats import entropy as scipy_entropy


def partition_entropy(parts, label_col='tier'):
    """Per-party Shannon entropy of label (tier) distribution (nats)."""
    entropies = []
    for p in parts:
        counts = p[label_col].value_counts().values
        entropies.append(scipy_entropy(counts))
    return np.array(entropies)


def summary_row(name, parts, label_col='tier'):
    sizes = np.array([len(p) for p in parts])
    ents  = partition_entropy(parts, label_col)
    global_counts = pd.concat(parts)[label_col].value_counts().values
    global_ent    = scipy_entropy(global_counts)
    return {
        'Strategy': name,
        'Sizes': sizes.tolist(),
        'Size std': round(float(sizes.std()), 1),
        'Min size': int(sizes.min()),
        'Max size': int(sizes.max()),
        'Label entropy MAD': round(float(np.mean(np.abs(ents - global_ent))), 4),
    }


# Label-based strategies use label_col='tier'; others fall back to 'tier' too
# (tier is available on all rows so entropy is meaningful across all methods)
rows = [
    summary_row('IID',
        IIDPartitioner(N_PARTIES, seed=42).partition(df)),
    summary_row('Qty-Label k=2',
        QuantityLabelPartitioner(N_PARTIES, n_labels_per_party=2, label_col='tier', seed=42).partition(df)),
    summary_row('Qty-Label k=3',
        QuantityLabelPartitioner(N_PARTIES, n_labels_per_party=3, label_col='tier', seed=42).partition(df)),
    summary_row('Dirichlet-Label α=0.1',
        DirichletLabelPartitioner(N_PARTIES, alpha=0.1, label_col='tier', seed=42).partition(df)),
    summary_row('Dirichlet-Label α=0.5',
        DirichletLabelPartitioner(N_PARTIES, alpha=0.5, label_col='tier', seed=42).partition(df)),
    summary_row('Dirichlet-Label α=5.0',
        DirichletLabelPartitioner(N_PARTIES, alpha=5.0, label_col='tier', seed=42).partition(df)),
    summary_row('Noise-Feature',
        NoiseFeaturePartitioner(N_PARTIES, seed=42).partition(df)),
    summary_row('Synthetic-Feature',
        SyntheticFeaturePartitioner(N_PARTIES, seed=42).partition(df)),
    summary_row('Qty-Skew α=0.1',
        QuantitySkewPartitioner(N_PARTIES, alpha=0.1, seed=42).partition(df)),
    summary_row('Qty-Skew α=0.5',
        QuantitySkewPartitioner(N_PARTIES, alpha=0.5, seed=42).partition(df)),
    summary_row('Qty-Skew α=5.0',
        QuantitySkewPartitioner(N_PARTIES, alpha=5.0, seed=42).partition(df)),
]

summary_df = pd.DataFrame(rows).set_index('Strategy')
print(summary_df[['Size std', 'Min size', 'Max size', 'Label entropy MAD']].to_string())
summary_df[['Size std', 'Min size', 'Max size', 'Label entropy MAD']]

In [ ]:
# Scatter: Size std vs Label entropy MAD — each strategy as a labelled point
fig, ax = plt.subplots(figsize=(10, 6))

colors_map = {
    'IID': 'black',
    'Qty-Label k=2': 'steelblue', 'Qty-Label k=4': 'cornflowerblue',
    'Dirichlet-Label α=0.1': 'crimson', 'Dirichlet-Label α=0.5': 'tomato',
    'Dirichlet-Label α=5.0': 'lightsalmon',
    'Noise-Feature': 'green',
    'Synthetic-Feature': 'purple',
    'Qty-Skew α=0.1': 'darkorange', 'Qty-Skew α=0.5': 'orange',
    'Qty-Skew α=5.0': 'moccasin',
}

for strategy, row in summary_df.iterrows():
    ax.scatter(row['Size std'], row['Label entropy MAD'],
               s=120, color=colors_map.get(strategy, 'gray'), zorder=5)
    ax.annotate(strategy, (row['Size std'], row['Label entropy MAD']),
                textcoords='offset points', xytext=(6, 3), fontsize=8)

ax.set_xlabel('Size std  (higher = more quantity skew)')
ax.set_ylabel('Label entropy MAD  (higher = more label skew)')
ax.set_title('Partitioning strategies: quantity skew vs label skew trade-off')
ax.axhline(0, color='grey', linewidth=0.5, linestyle='--')
ax.axvline(0, color='grey', linewidth=0.5, linestyle='--')
plt.tight_layout()
plt.show()